# Weekly external volatility-context features 

This notebook adds external market-volatility context to the existing weekly stock-direction modelling dataset. It keeps the original weekly notebooks unchanged and writes a separate reproducible context dataset under `outputs/weekly_context/`.


The target remains `Target_State_Binary`, where `1` means next-week return is positive and `0` means next-week return is non-positive.

## Motivation

The external volatility features are motivated by the idea that individual stock returns contain both systematic market-driven components and idiosyncratic components. VIX provides a broad market fear signal, while VXN provides a Nasdaq-specific fear signal that may be more relevant for technology-heavy stocks. These variables may help the model distinguish whether a stock's recent movement is occurring under calm market conditions or broad risk-off conditions.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

RANDOM_STATE = 5

WEEKLY_INPUT_PATH = Path("outputs/weekly/model_df_weekly.parquet")
CONTEXT_OUTPUT_DIR = Path("outputs/weekly_context")
PLOTS_DIR = CONTEXT_OUTPUT_DIR / "plots"

EXTERNAL_TICKERS = {
    "VIX": "^VIX",
    "VXN": "^VXN",
}

BASELINE_WEEKLY_FEATURES = [
    "Weekly_Log_Return",
    "Weekly_Open_Close_Log_Return",
    "Weekly_High_Low_Range",
    "Weekly_Volume_Change",
    "Rolling_Vol_4",
    "Rolling_Vol_12",
    "Rolling_Vol_26",
    "Momentum_4",
    "Momentum_12",
    "Momentum_26",
    "MA_Gap_4",
    "MA_Gap_12",
    "MA_Gap_26",
    "Drawdown_12",
    "Drawdown_26",
]

CORE_VOL_CONTEXT_FEATURES = [
    "VIX_Close",
    "VIX_Change",
    "VIX_Log_Change",
    "VIX_MA_4",
    "VIX_MA_12",
    "VIX_ZScore_12",
    "VIX_Above_MA_12",
    "VXN_Close",
    "VXN_Change",
    "VXN_Log_Change",
    "VXN_MA_4",
    "VXN_MA_12",
    "VXN_ZScore_12",
    "VXN_Above_MA_12",
]

RELATIVE_FEAR_FEATURES = [
    "VXN_minus_VIX",
    "VXN_to_VIX",
]

LEAKAGE_COLUMNS = {
    "Target_State_Binary",
    "Target_State",
    "Next_Week_Log_Return",
    "Target_Date",
    "Ticker",
    "Date",
    "Split",
}

CONTEXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Context output directory:", CONTEXT_OUTPUT_DIR)
print("Plots directory:", PLOTS_DIR)

Context output directory: outputs\weekly_context
Plots directory: outputs\weekly_context\plots


## 1. Load weekly modelling data

The external volatility download uses the minimum and maximum `Date` already present in `outputs/weekly/model_df_weekly.parquet`. These dates are week-ending Fridays in the current weekly modelling dataset.

In [2]:
if not WEEKLY_INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing weekly modelling data: {WEEKLY_INPUT_PATH}")

model_df_weekly_original = pd.read_parquet(WEEKLY_INPUT_PATH).copy()
model_df_weekly_original["Date"] = pd.to_datetime(model_df_weekly_original["Date"])
if "Target_Date" in model_df_weekly_original.columns:
    model_df_weekly_original["Target_Date"] = pd.to_datetime(model_df_weekly_original["Target_Date"])

missing_baseline_features = [col for col in BASELINE_WEEKLY_FEATURES if col not in model_df_weekly_original.columns]
if missing_baseline_features:
    raise ValueError(f"Missing baseline weekly features: {missing_baseline_features}")

required_columns = ["Ticker", "Date", "Target_Date", "Split", "Next_Week_Log_Return", "Target_State_Binary", "Target_State"]
missing_required_columns = [col for col in required_columns if col not in model_df_weekly_original.columns]
if missing_required_columns:
    raise ValueError(f"Missing required weekly columns: {missing_required_columns}")

model_start_date = model_df_weekly_original["Date"].min()
model_end_date = model_df_weekly_original["Date"].max()
download_end_date = model_end_date + pd.Timedelta(days=1)

print("Original weekly model dataframe shape:", model_df_weekly_original.shape)
print("Weekly model Date range:", model_start_date.date(), "to", model_end_date.date())
print("Download end date passed to yfinance:", download_end_date.date(), "(exclusive)")
display(model_df_weekly_original.groupby(["Ticker", "Split"]).size().unstack(fill_value=0))

Original weekly model dataframe shape: (2424, 29)
Weekly model Date range: 2010-07-09 to 2025-12-26
Download end date passed to yfinance: 2025-12-27 (exclusive)


Split,test,train,validation
Ticker,,,
AAPL,122,565,121
IBM,122,565,121
MSFT,122,565,121


## 2. Download daily VIX and VXN data

The notebook uses `yfinance` for `^VIX` and `^VXN`. Only daily close values are used.

In [3]:
def extract_close_series(downloaded: pd.DataFrame, label: str) -> pd.Series:
    if downloaded.empty:
        raise ValueError(f"No rows downloaded for {label}.")

    if isinstance(downloaded.columns, pd.MultiIndex):
        level_0 = downloaded.columns.get_level_values(0)
        if "Close" in level_0:
            close_obj = downloaded["Close"]
        elif "Adj Close" in level_0:
            close_obj = downloaded["Adj Close"]
        else:
            raise ValueError(f"Could not find Close or Adj Close column for {label}.")

        if isinstance(close_obj, pd.DataFrame):
            close_series = close_obj.iloc[:, 0]
        else:
            close_series = close_obj
    else:
        if "Close" in downloaded.columns:
            close_series = downloaded["Close"]
        elif "Adj Close" in downloaded.columns:
            close_series = downloaded["Adj Close"]
        else:
            raise ValueError(f"Could not find Close or Adj Close column for {label}.")

    close_series = close_series.astype(float).rename(f"{label}_Close")
    close_series.index = pd.to_datetime(close_series.index).tz_localize(None)
    return close_series.sort_index()


daily_close_series = []
download_status_rows = []

for label, symbol in EXTERNAL_TICKERS.items():
    raw_download = yf.download(
        symbol,
        start=model_start_date.strftime("%Y-%m-%d"),
        end=download_end_date.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        progress=False,
    )
    close_series = extract_close_series(raw_download, label)
    daily_close_series.append(close_series)
    download_status_rows.append(
        {
            "Label": label,
            "Symbol": symbol,
            "Rows": int(close_series.shape[0]),
            "First_Date": close_series.index.min(),
            "Last_Date": close_series.index.max(),
            "Downloaded": bool(close_series.shape[0] > 0),
        }
    )

external_daily_close_df = pd.concat(daily_close_series, axis=1).sort_index()
download_status_df = pd.DataFrame(download_status_rows)

if external_daily_close_df.empty:
    raise ValueError("No external volatility data was downloaded.")

print("Daily external volatility close shape:", external_daily_close_df.shape)
display(download_status_df)
display(external_daily_close_df.head())
display(external_daily_close_df.tail())

Daily external volatility close shape: (3892, 2)


,Label,Symbol,Rows,First_Date,Last_Date,Downloaded
0,VIX,^VIX,3892,2010-07-09,2025-12-26,True
1,VXN,^VXN,3892,2010-07-09,2025-12-26,True


,VIX_Close,VXN_Close
Date,,
2010-07-09,24.980000,25.990000
2010-07-12,24.430000,25.500000
2010-07-13,24.559999,25.370001
2010-07-14,24.889999,25.959999
2010-07-15,25.139999,26.570000


,VIX_Close,VXN_Close
Date,,
2025-12-19,14.91,18.520000
2025-12-22,14.08,17.850000
2025-12-23,14.00,17.469999
2025-12-24,13.47,17.200001
2025-12-26,13.60,17.379999


## 3. Aggregate to weekly W-FRI features

Daily VIX and VXN values are converted to weekly values with `resample("W-FRI").last()`. Rolling and differenced features use only the current week and earlier weeks, so the features are measured at week `t` and can be used for next-week classification at `t+1`.

In [4]:
WEEKLY_RESAMPLE_RULE = "W-FRI"

external_weekly_close_df = external_daily_close_df.resample(WEEKLY_RESAMPLE_RULE).last()
external_weekly_features_df = external_weekly_close_df.copy()

for label in EXTERNAL_TICKERS:
    close_col = f"{label}_Close"
    close = external_weekly_features_df[close_col]

    external_weekly_features_df[f"{label}_Change"] = close.diff()
    external_weekly_features_df[f"{label}_Log_Change"] = np.log(close).diff()
    external_weekly_features_df[f"{label}_MA_4"] = close.rolling(4).mean()
    external_weekly_features_df[f"{label}_MA_12"] = close.rolling(12).mean()
    external_weekly_features_df[f"{label}_ZScore_12"] = (
        (close - external_weekly_features_df[f"{label}_MA_12"])
        / close.rolling(12).std()
    )
    external_weekly_features_df[f"{label}_Above_MA_12"] = (close > external_weekly_features_df[f"{label}_MA_12"]).astype(int)

external_weekly_features_df["VXN_minus_VIX"] = external_weekly_features_df["VXN_Close"] - external_weekly_features_df["VIX_Close"]
external_weekly_features_df["VXN_to_VIX"] = external_weekly_features_df["VXN_Close"] / external_weekly_features_df["VIX_Close"]

external_weekly_features_df = external_weekly_features_df.replace([np.inf, -np.inf], np.nan)
external_weekly_features_df = external_weekly_features_df.reset_index().rename(columns={"index": "Date"})
external_weekly_features_df["Date"] = pd.to_datetime(external_weekly_features_df["Date"])

print("Weekly external volatility feature shape before merging:", external_weekly_features_df.shape)
print("Weekly resample rule:", WEEKLY_RESAMPLE_RULE)
display(external_weekly_features_df.head(15))

Weekly external volatility feature shape before merging: (808, 17)
Weekly resample rule: W-FRI


,Date,VIX_Close,VXN_Close,VIX_Change,VIX_Log_Change,VIX_MA_4,VIX_MA_12,VIX_ZScore_12,VIX_Above_MA_12,VXN_Change,VXN_Log_Change,VXN_MA_4,VXN_MA_12,VXN_ZScore_12,VXN_Above_MA_12,VXN_minus_VIX,VXN_to_VIX
0,2010-07-09,24.980000,25.990000,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,1.010000,1.040432
1,2010-07-16,26.250000,26.870001,1.270000,0.049591,NaN,NaN,NaN,0,0.880001,0.033299,NaN,NaN,NaN,0,0.620001,1.023619
2,2010-07-23,23.469999,24.160000,-2.780001,-0.111943,NaN,NaN,NaN,0,-2.710001,-0.106312,NaN,NaN,NaN,0,0.690001,1.029399
3,2010-07-30,23.500000,24.520000,0.030001,0.001277,24.5500,NaN,NaN,0,0.360001,0.014791,25.3850,NaN,NaN,0,1.020000,1.043404
4,2010-08-06,21.740000,22.840000,-1.760000,-0.077847,23.7400,NaN,NaN,0,-1.680000,-0.070976,24.5975,NaN,NaN,0,1.100000,1.050598
5,2010-08-13,26.240000,28.070000,4.500000,0.188131,23.7375,NaN,NaN,0,5.230000,0.206188,24.8975,NaN,NaN,0,1.830000,1.069741
6,2010-08-20,25.490000,25.760000,-0.750000,-0.028999,24.2425,NaN,NaN,0,-2.309999,-0.085878,25.2975,NaN,NaN,0,0.270000,1.010592
7,2010-08-27,24.450001,26.309999,-1.039999,-0.041656,24.4800,NaN,NaN,0,0.549999,0.021126,25.7450,NaN,NaN,0,1.859999,1.076074
8,2010-09-03,21.309999,21.950001,-3.140001,-0.137454,24.3725,NaN,NaN,0,-4.359999,-0.181182,25.5225,NaN,NaN,0,0.640001,1.030033
9,2010-09-10,21.990000,22.980000,0.680000,0.031411,23.3100,NaN,NaN,0,1.029999,0.045857,24.2500,NaN,NaN,0,0.990000,1.045020


## 4. Merge external features by Date only

VIX and VXN are market-level features, so every ticker-week receives the same volatility-context values for a given `Date`. The merge is `many_to_one` from ticker rows to weekly volatility rows.

In [5]:
external_feature_cols_for_merge = CORE_VOL_CONTEXT_FEATURES + RELATIVE_FEAR_FEATURES
missing_external_cols = [col for col in external_feature_cols_for_merge if col not in external_weekly_features_df.columns]
if missing_external_cols:
    raise ValueError(f"Missing external feature columns before merge: {missing_external_cols}")

model_df_weekly_merged = model_df_weekly_original.merge(
    external_weekly_features_df[["Date"] + external_feature_cols_for_merge],
    on="Date",
    how="left",
    validate="many_to_one",
)
model_df_weekly_merged = model_df_weekly_merged.replace([np.inf, -np.inf], np.nan)

core_context_missing_mask = model_df_weekly_merged[CORE_VOL_CONTEXT_FEATURES].isna().any(axis=1)
rows_with_initial_context_missing = int(core_context_missing_mask.sum())

model_df_weekly_with_context = model_df_weekly_merged.loc[~core_context_missing_mask].copy()
model_df_weekly_with_context = model_df_weekly_with_context.sort_values(["Ticker", "Date"]).reset_index(drop=True)

relative_values = model_df_weekly_with_context[RELATIVE_FEAR_FEATURES]
relative_fear_features_are_stable = bool(
    relative_values.notna().all().all()
    and np.isfinite(relative_values.to_numpy(dtype=float)).all()
    and (model_df_weekly_with_context["VIX_Close"] != 0).all()
)

if relative_fear_features_are_stable:
    VOL_CONTEXT_FEATURES = CORE_VOL_CONTEXT_FEATURES + RELATIVE_FEAR_FEATURES
else:
    VOL_CONTEXT_FEATURES = CORE_VOL_CONTEXT_FEATURES.copy()
    model_df_weekly_with_context = model_df_weekly_with_context.drop(columns=RELATIVE_FEAR_FEATURES, errors="ignore")

CONTEXT_AWARE_FEATURES = BASELINE_WEEKLY_FEATURES + VOL_CONTEXT_FEATURES

feature_leakage = sorted(set(CONTEXT_AWARE_FEATURES).intersection(LEAKAGE_COLUMNS))
if feature_leakage:
    raise ValueError(f"Leakage columns found in model feature list: {feature_leakage}")

missing_context_after_filter = model_df_weekly_with_context[CONTEXT_AWARE_FEATURES].isna().sum()
if int(missing_context_after_filter.sum()) != 0:
    raise ValueError("Feature columns still contain missing values after external-context filtering.")

expected_feature_count = len(BASELINE_WEEKLY_FEATURES) + len(VOL_CONTEXT_FEATURES)
print("Rows before context filtering:", model_df_weekly_original.shape[0])
print("Rows dropped due to initial weekly external rolling/diff missing values:", rows_with_initial_context_missing)
print("Rows after context filtering:", model_df_weekly_with_context.shape[0])
print("VOL_CONTEXT_FEATURES:", VOL_CONTEXT_FEATURES)
print("Context-aware feature count:", expected_feature_count)
print("Relative fear features included:", relative_fear_features_are_stable)

date_only_consistency = (
    model_df_weekly_with_context
    .groupby("Date")[VOL_CONTEXT_FEATURES]
    .nunique(dropna=False)
    .le(1)
    .all()
    .all()
)
if not date_only_consistency:
    raise ValueError("At least one external volatility feature differs across tickers for the same Date.")

display(model_df_weekly_with_context[["Ticker", "Date", "Split", "Target_State_Binary"] + VOL_CONTEXT_FEATURES].head())
display(model_df_weekly_with_context.groupby(["Ticker", "Split"]).size().unstack(fill_value=0))

Rows before context filtering: 2424
Rows dropped due to initial weekly external rolling/diff missing values: 33
Rows after context filtering: 2391
VOL_CONTEXT_FEATURES: ['VIX_Close', 'VIX_Change', 'VIX_Log_Change', 'VIX_MA_4', 'VIX_MA_12', 'VIX_ZScore_12', 'VIX_Above_MA_12', 'VXN_Close', 'VXN_Change', 'VXN_Log_Change', 'VXN_MA_4', 'VXN_MA_12', 'VXN_ZScore_12', 'VXN_Above_MA_12', 'VXN_minus_VIX', 'VXN_to_VIX']
Context-aware feature count: 31
Relative fear features included: True


,Ticker,Date,Split,Target_State_Binary,VIX_Close,VIX_Change,VIX_Log_Change,VIX_MA_4,VIX_MA_12,VIX_ZScore_12,VIX_Above_MA_12,VXN_Close,VXN_Change,VXN_Log_Change,VXN_MA_4,VXN_MA_12,VXN_ZScore_12,VXN_Above_MA_12,VXN_minus_VIX,VXN_to_VIX
0,AAPL,2010-09-24,train,0,21.709999,-0.300001,-0.013724,21.7550,23.595000,-1.019377,0,22.719999,0.459999,0.020454,22.4775,24.535833,-0.894788,0,1.010000,1.046522
1,AAPL,2010-10-01,train,1,22.500000,0.790001,0.035742,22.0525,23.388333,-0.488460,0,23.980000,1.260000,0.053975,22.9850,24.368333,-0.196051,0,1.480000,1.065778
2,AAPL,2010-10-08,train,1,20.709999,-1.790001,-0.082899,21.7325,22.926666,-1.283520,0,21.980000,-2.000000,-0.087087,22.7350,23.960833,-1.030908,0,1.270000,1.061323
3,AAPL,2010-10-15,train,0,19.030001,-1.679998,-0.084600,20.9875,22.556667,-1.723550,0,20.440001,-1.539999,-0.072639,22.2800,23.650833,-1.479404,0,1.410000,1.074094
4,AAPL,2010-10-22,train,0,18.780001,-0.250000,-0.013224,20.2550,22.163333,-1.478895,0,20.420000,-0.020000,-0.000979,21.7050,23.309167,-1.236070,0,1.639999,1.087327


Split,test,train,validation
Ticker,,,
AAPL,122,554,121
IBM,122,554,121
MSFT,122,554,121


## 5. Save enriched unscaled context dataset

In [6]:
context_parquet_path = CONTEXT_OUTPUT_DIR / "model_df_weekly_with_vol_context.parquet"
context_csv_path = CONTEXT_OUTPUT_DIR / "model_df_weekly_with_vol_context.csv"

model_df_weekly_with_context.to_parquet(context_parquet_path, index=False)
model_df_weekly_with_context.to_csv(context_csv_path, index=False)

for output_path in [context_parquet_path, context_csv_path]:
    if not output_path.exists():
        raise FileNotFoundError(f"Expected output was not saved: {output_path}")

print("Saved enriched context dataset:")
print(context_parquet_path)
print(context_csv_path)

Saved enriched context dataset:
outputs\weekly_context\model_df_weekly_with_vol_context.parquet
outputs\weekly_context\model_df_weekly_with_vol_context.csv


## 6. Train-only scaling without leakage

The original 15 stock-specific weekly features are scaled with the same pattern as `LSTM_input_weekly.ipynb`: one `StandardScaler` per ticker, fitted on training rows only.

The VIX/VXN context features are market-level features shared across tickers. They are scaled with a global market-context `StandardScaler`, also fitted only on rows where `Split == "train"`.

In [7]:
model_df_weekly_with_context_scaled = model_df_weekly_with_context.copy()
model_df_weekly_with_context_scaled[BASELINE_WEEKLY_FEATURES + VOL_CONTEXT_FEATURES] = model_df_weekly_with_context_scaled[
    BASELINE_WEEKLY_FEATURES + VOL_CONTEXT_FEATURES
].astype(float)
scaler_metadata_rows = []

for ticker, group in model_df_weekly_with_context.groupby("Ticker", sort=True):
    train_mask = group["Split"] == "train"
    if not train_mask.any():
        raise ValueError(f"Ticker {ticker} has no training rows for baseline scaler fitting.")

    scaler = StandardScaler()
    scaler.fit(group.loc[train_mask, BASELINE_WEEKLY_FEATURES])
    transformed = scaler.transform(group[BASELINE_WEEKLY_FEATURES])
    model_df_weekly_with_context_scaled.loc[group.index, BASELINE_WEEKLY_FEATURES] = transformed

    for feature, mean, scale in zip(BASELINE_WEEKLY_FEATURES, scaler.mean_, scaler.scale_):
        scaler_metadata_rows.append(
            {
                "Scaler_Scope": "ticker_stock_features",
                "Ticker": ticker,
                "Feature": feature,
                "Train_Mean": mean,
                "Train_Scale": scale,
            }
        )

context_train_mask = model_df_weekly_with_context["Split"] == "train"
if not context_train_mask.any():
    raise ValueError("No training rows available for global market-context scaler fitting.")

context_scaler = StandardScaler()
context_scaler.fit(model_df_weekly_with_context.loc[context_train_mask, VOL_CONTEXT_FEATURES])
model_df_weekly_with_context_scaled.loc[:, VOL_CONTEXT_FEATURES] = context_scaler.transform(
    model_df_weekly_with_context[VOL_CONTEXT_FEATURES]
)

for feature, mean, scale in zip(VOL_CONTEXT_FEATURES, context_scaler.mean_, context_scaler.scale_):
    scaler_metadata_rows.append(
        {
            "Scaler_Scope": "global_market_context_features",
            "Ticker": "ALL",
            "Feature": feature,
            "Train_Mean": mean,
            "Train_Scale": scale,
        }
    )

vol_context_scaler_metadata_df = pd.DataFrame(scaler_metadata_rows)

leakage_scaled = sorted(set(CONTEXT_AWARE_FEATURES).intersection(LEAKAGE_COLUMNS))
if leakage_scaled:
    raise ValueError(f"Leakage columns found in scaled feature list: {leakage_scaled}")

scaled_context_parquet_path = CONTEXT_OUTPUT_DIR / "model_df_weekly_with_vol_context_scaled.parquet"
scaled_context_csv_path = CONTEXT_OUTPUT_DIR / "model_df_weekly_with_vol_context_scaled.csv"
scaler_metadata_path = CONTEXT_OUTPUT_DIR / "vol_context_scaler_metadata.csv"

model_df_weekly_with_context_scaled.to_parquet(scaled_context_parquet_path, index=False)
model_df_weekly_with_context_scaled.to_csv(scaled_context_csv_path, index=False)
vol_context_scaler_metadata_df.to_csv(scaler_metadata_path, index=False)

for output_path in [scaled_context_parquet_path, scaled_context_csv_path, scaler_metadata_path]:
    if not output_path.exists():
        raise FileNotFoundError(f"Expected output was not saved: {output_path}")

print("Saved scaled context dataset and scaler metadata:")
print(scaled_context_parquet_path)
print(scaled_context_csv_path)
print(scaler_metadata_path)

display(vol_context_scaler_metadata_df.head())
print("Scaled training means for stock-specific features by ticker:")
display(
    model_df_weekly_with_context_scaled
    .loc[model_df_weekly_with_context_scaled["Split"] == "train"]
    .groupby("Ticker")[BASELINE_WEEKLY_FEATURES]
    .mean()
    .round(4)
)
print("Scaled training means for global market context features:")
display(
    model_df_weekly_with_context_scaled
    .loc[model_df_weekly_with_context_scaled["Split"] == "train", VOL_CONTEXT_FEATURES]
    .mean()
    .round(4)
)

Saved scaled context dataset and scaler metadata:
outputs\weekly_context\model_df_weekly_with_vol_context_scaled.parquet
outputs\weekly_context\model_df_weekly_with_vol_context_scaled.csv
outputs\weekly_context\vol_context_scaler_metadata.csv


,Scaler_Scope,Ticker,Feature,Train_Mean,Train_Scale
0,ticker_stock_features,AAPL,Weekly_Log_Return,0.004950,0.038727
1,ticker_stock_features,AAPL,Weekly_Open_Close_Log_Return,0.004503,0.036854
2,ticker_stock_features,AAPL,Weekly_High_Low_Range,0.051066,0.027829
3,ticker_stock_features,AAPL,Weekly_Volume_Change,-0.002909,0.341152
4,ticker_stock_features,AAPL,Rolling_Vol_4,0.034025,0.018935


Scaled training means for stock-specific features by ticker:


,Weekly_Log_Return,Weekly_Open_Close_Log_Return,Weekly_High_Low_Range,Weekly_Volume_Change,Rolling_Vol_4,Rolling_Vol_12,Rolling_Vol_26,Momentum_4,Momentum_12,Momentum_26,MA_Gap_4,MA_Gap_12,MA_Gap_26,Drawdown_12,Drawdown_26
Ticker,,,,,,,,,,,,,,,
AAPL,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0
IBM,0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0
MSFT,0.0,0.0,-0.0,0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0


Scaled training means for global market context features:


VIX_Close         -0.0
VIX_Change        -0.0
VIX_Log_Change     0.0
VIX_MA_4           0.0
VIX_MA_12          0.0
VIX_ZScore_12     -0.0
VIX_Above_MA_12    0.0
VXN_Close         -0.0
VXN_Change         0.0
VXN_Log_Change     0.0
VXN_MA_4          -0.0
VXN_MA_12         -0.0
VXN_ZScore_12     -0.0
VXN_Above_MA_12    0.0
VXN_minus_VIX     -0.0
VXN_to_VIX         0.0
dtype: float64

## 7. Exploratory diagnostics

All plots use matplotlib only and are saved under `outputs/weekly_context/plots/`.

In [8]:
def save_current_figure(filename: str) -> Path:
    path = PLOTS_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()
    return path

plot_paths = []
unique_weekly_context_df = (
    model_df_weekly_with_context
    .drop_duplicates("Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

plt.figure(figsize=(12, 5))
plt.plot(unique_weekly_context_df["Date"], unique_weekly_context_df["VIX_Close"], label="VIX weekly close", linewidth=1.5)
plt.plot(unique_weekly_context_df["Date"], unique_weekly_context_df["VXN_Close"], label="VXN weekly close", linewidth=1.5)
plt.title("VIX and VXN weekly close")
plt.xlabel("Date")
plt.ylabel("Weekly close")
plt.legend()
plt.grid(alpha=0.25)
plot_paths.append(save_current_figure("vix_vxn_weekly_close.png"))

plt.figure(figsize=(12, 5))
plt.plot(unique_weekly_context_df["Date"], unique_weekly_context_df["VIX_ZScore_12"], label="VIX 12-week z-score", linewidth=1.3)
plt.plot(unique_weekly_context_df["Date"], unique_weekly_context_df["VXN_ZScore_12"], label="VXN 12-week z-score", linewidth=1.3)
plt.axhline(0, color="black", linewidth=0.8, alpha=0.6)
plt.title("VIX/VXN 12-week z-scores")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend()
plt.grid(alpha=0.25)
plot_paths.append(save_current_figure("vix_vxn_zscores.png"))

if "VXN_minus_VIX" in model_df_weekly_with_context.columns:
    plt.figure(figsize=(12, 5))
    plt.plot(unique_weekly_context_df["Date"], unique_weekly_context_df["VXN_minus_VIX"], color="tab:purple", linewidth=1.4)
    plt.axhline(0, color="black", linewidth=0.8, alpha=0.6)
    plt.title("VXN minus VIX over time")
    plt.xlabel("Date")
    plt.ylabel("VXN - VIX")
    plt.grid(alpha=0.25)
    plot_paths.append(save_current_figure("vxn_minus_vix.png"))

rng = np.random.default_rng(RANDOM_STATE)
scatter_df = model_df_weekly_with_context[["VIX_Change", "Target_State_Binary"]].dropna().copy()
scatter_y = scatter_df["Target_State_Binary"].to_numpy(dtype=float) + rng.uniform(-0.04, 0.04, size=scatter_df.shape[0])
plt.figure(figsize=(9, 5))
plt.scatter(scatter_df["VIX_Change"], scatter_y, alpha=0.35, s=18)
plt.yticks([0, 1], ["Bearish/0", "Bullish/1"])
plt.title("VIX weekly change vs next-week target class")
plt.xlabel("VIX weekly change")
plt.ylabel("Next-week target class")
plt.grid(alpha=0.25)
plot_paths.append(save_current_figure("vix_change_vs_target_scatter.png"))

boxplot_groups = [
    model_df_weekly_with_context.loc[model_df_weekly_with_context["Target_State_Binary"] == label, "VXN_Change"].dropna().to_numpy()
    for label in [0, 1]
]
plt.figure(figsize=(8, 5))
plt.boxplot(boxplot_groups, labels=["Bearish/0", "Bullish/1"], showfliers=False)
plt.title("VXN weekly change by next-week target class")
plt.xlabel("Next-week target class")
plt.ylabel("VXN weekly change")
plt.grid(axis="y", alpha=0.25)
plot_paths.append(save_current_figure("vxn_change_by_target_boxplot.png"))

correlation_cols = VOL_CONTEXT_FEATURES + ["Target_State_Binary"]
vol_context_correlation_df = model_df_weekly_with_context[correlation_cols].corr(numeric_only=True)
correlation_table_path = CONTEXT_OUTPUT_DIR / "vol_context_target_correlation.csv"
vol_context_correlation_df.to_csv(correlation_table_path)

fig_width = max(10, 0.45 * len(correlation_cols))
fig_height = max(8, 0.45 * len(correlation_cols))
plt.figure(figsize=(fig_width, fig_height))
plt.imshow(vol_context_correlation_df, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation_cols)), correlation_cols, rotation=90)
plt.yticks(range(len(correlation_cols)), correlation_cols)
plt.title("Correlation: VIX/VXN features and next-week target")
plot_paths.append(save_current_figure("vol_context_target_correlation_heatmap.png"))

plot_paths.append(correlation_table_path)

print("Saved diagnostic plot/table files:")
for plot_path in plot_paths:
    print(plot_path)

C:\Users\Admin\AppData\Local\Temp\ipykernel_26204\3078222774.py:64: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(boxplot_groups, labels=["Bearish/0", "Bullish/1"], showfliers=False)


Saved diagnostic plot/table files:
outputs\weekly_context\plots\vix_vxn_weekly_close.png
outputs\weekly_context\plots\vix_vxn_zscores.png
outputs\weekly_context\plots\vxn_minus_vix.png
outputs\weekly_context\plots\vix_change_vs_target_scatter.png
outputs\weekly_context\plots\vxn_change_by_target_boxplot.png
outputs\weekly_context\plots\vol_context_target_correlation_heatmap.png
outputs\weekly_context\vol_context_target_correlation.csv


## 8. Feature sets

These feature sets are prepared for later modelling experiments. This notebook intentionally does not replace the main HMM-LSTM results and does not train the LSTM comparison models.

In [9]:
print("BASELINE_WEEKLY_FEATURES", len(BASELINE_WEEKLY_FEATURES))
print(BASELINE_WEEKLY_FEATURES)
print("\nVOL_CONTEXT_FEATURES", len(VOL_CONTEXT_FEATURES))
print(VOL_CONTEXT_FEATURES)
print("\nCONTEXT_AWARE_FEATURES", len(CONTEXT_AWARE_FEATURES))
print(CONTEXT_AWARE_FEATURES)

if len(BASELINE_WEEKLY_FEATURES) != 15:
    raise ValueError(f"Expected 15 baseline weekly features, found {len(BASELINE_WEEKLY_FEATURES)}")

expected_context_feature_count = 31 if relative_fear_features_are_stable else 29
if len(CONTEXT_AWARE_FEATURES) != expected_context_feature_count:
    raise ValueError(
        f"Expected {expected_context_feature_count} context-aware features, found {len(CONTEXT_AWARE_FEATURES)}"
    )

BASELINE_WEEKLY_FEATURES 15
['Weekly_Log_Return', 'Weekly_Open_Close_Log_Return', 'Weekly_High_Low_Range', 'Weekly_Volume_Change', 'Rolling_Vol_4', 'Rolling_Vol_12', 'Rolling_Vol_26', 'Momentum_4', 'Momentum_12', 'Momentum_26', 'MA_Gap_4', 'MA_Gap_12', 'MA_Gap_26', 'Drawdown_12', 'Drawdown_26']

VOL_CONTEXT_FEATURES 16
['VIX_Close', 'VIX_Change', 'VIX_Log_Change', 'VIX_MA_4', 'VIX_MA_12', 'VIX_ZScore_12', 'VIX_Above_MA_12', 'VXN_Close', 'VXN_Change', 'VXN_Log_Change', 'VXN_MA_4', 'VXN_MA_12', 'VXN_ZScore_12', 'VXN_Above_MA_12', 'VXN_minus_VIX', 'VXN_to_VIX']

CONTEXT_AWARE_FEATURES 31
['Weekly_Log_Return', 'Weekly_Open_Close_Log_Return', 'Weekly_High_Low_Range', 'Weekly_Volume_Change', 'Rolling_Vol_4', 'Rolling_Vol_12', 'Rolling_Vol_26', 'Momentum_4', 'Momentum_12', 'Momentum_26', 'MA_Gap_4', 'MA_Gap_12', 'MA_Gap_26', 'Drawdown_12', 'Drawdown_26', 'VIX_Close', 'VIX_Change', 'VIX_Log_Change', 'VIX_MA_4', 'VIX_MA_12', 'VIX_ZScore_12', 'VIX_Above_MA_12', 'VXN_Close', 'VXN_Change', 'VXN_Lo

## 9. Final sanity checks

In [10]:
expected_filtered_row_count = int((~core_context_missing_mask).sum())
required_output_paths = [
    context_parquet_path,
    context_csv_path,
    scaled_context_parquet_path,
    scaled_context_csv_path,
    scaler_metadata_path,
]
required_plot_paths = [
    PLOTS_DIR / "vix_vxn_weekly_close.png",
    PLOTS_DIR / "vix_vxn_zscores.png",
    PLOTS_DIR / "vxn_minus_vix.png",
    PLOTS_DIR / "vix_change_vs_target_scatter.png",
    PLOTS_DIR / "vxn_change_by_target_boxplot.png",
    PLOTS_DIR / "vol_context_target_correlation_heatmap.png",
    CONTEXT_OUTPUT_DIR / "vol_context_target_correlation.csv",
]

if "VXN_minus_VIX" not in model_df_weekly_with_context.columns:
    required_plot_paths.remove(PLOTS_DIR / "vxn_minus_vix.png")

sanity_checks = {
    "VIX and VXN were downloaded successfully": bool(download_status_df["Downloaded"].all()),
    "Weekly aggregation uses W-FRI": WEEKLY_RESAMPLE_RULE == "W-FRI",
    "External features are aligned by Date only": bool(date_only_consistency),
    "No future VIX/VXN values are used": WEEKLY_RESAMPLE_RULE == "W-FRI",
    "Enriched rows equal original rows after consistent initial rolling-value filtering": model_df_weekly_with_context.shape[0] == expected_filtered_row_count,
    "Train-only scaling is used": set(vol_context_scaler_metadata_df["Scaler_Scope"]) == {"ticker_stock_features", "global_market_context_features"},
    "Target/date/ticker/split columns are not included as features": not set(CONTEXT_AWARE_FEATURES).intersection(LEAKAGE_COLUMNS),
    "Output parquet and CSV files exist": all(path.exists() for path in required_output_paths),
    "Plot files exist": all(path.exists() for path in required_plot_paths),
}

for label, passed in sanity_checks.items():
    status = "OK" if passed else "FAIL"
    print(f"[{status}] {label}")

if not all(sanity_checks.values()):
    failed = [label for label, passed in sanity_checks.items() if not passed]
    raise AssertionError(f"Final sanity checks failed: {failed}")

print("\nOriginal row count:", model_df_weekly_original.shape[0])
print("Context row count after initial external rolling filters:", model_df_weekly_with_context.shape[0])
print("Dropped rows:", model_df_weekly_original.shape[0] - model_df_weekly_with_context.shape[0])
print("Feature count:", len(CONTEXT_AWARE_FEATURES))
print("Saved files:")
for path in required_output_paths + required_plot_paths:
    print(path)

[OK] VIX and VXN were downloaded successfully
[OK] Weekly aggregation uses W-FRI
[OK] External features are aligned by Date only
[OK] No future VIX/VXN values are used
[OK] Enriched rows equal original rows after consistent initial rolling-value filtering
[OK] Train-only scaling is used
[OK] Target/date/ticker/split columns are not included as features
[OK] Output parquet and CSV files exist
[OK] Plot files exist

Original row count: 2424
Context row count after initial external rolling filters: 2391
Dropped rows: 33
Feature count: 31
Saved files:
outputs\weekly_context\model_df_weekly_with_vol_context.parquet
outputs\weekly_context\model_df_weekly_with_vol_context.csv
outputs\weekly_context\model_df_weekly_with_vol_context_scaled.parquet
outputs\weekly_context\model_df_weekly_with_vol_context_scaled.csv
outputs\weekly_context\vol_context_scaler_metadata.csv
outputs\weekly_context\plots\vix_vxn_weekly_close.png
outputs\weekly_context\plots\vix_vxn_zscores.png
outputs\weekly_context\plo